# 階層型GP: 条件によって有効になる変数が変わる場合

このNotebookでは `HierarchicalConditionalKernelGP` と `HierarchicalConditionalKernelMultiTaskGP` を使います。

例として、共通条件 `x` に加えて、プロセス方式によって有効な条件が変わる探索空間を考えます。

- 方式0: 温度が有効、圧力は無効
- 方式1: 圧力が有効、温度は無効

通常のGPでは無効な変数まで距離計算に入ってしまいますが、階層型カーネルでは親変数の値に応じて有効な子変数だけを使って共分散を計算できます。

## 1. Import と再現性設定

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll
from robotorchan.models import (
    HierarchicalConditionalKernelGP,
    HierarchicalConditionalKernelMultiTaskGP,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. 階層構造を定義する

入力を `[x, process_type, temperature, pressure]` とします。

`hierarchical_dependencies` は `{親の列: {親の値: [有効になる子の列]}}` という形式です。

したがって、方式0では温度、方式1では圧力が有効になる構造は次のように表現できます。

In [ ]:
hierarchical_dependencies = {
    1: {
        0: [2],  # process_type == 0 -> temperature が有効
        1: [3],  # process_type == 1 -> pressure が有効
    }
}
hierarchical_dependencies

## 3. Synthetic data

共通変数 `x` はどちらの方式でも効きます。方式0では温度、方式1では圧力だけが応答に影響するようにします。

無効な変数にも数値は入っていますが、階層型カーネルではその枝で無効な変数として扱われます。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    x = X[..., 0]
    process_type = X[..., 1]
    temperature = X[..., 2]
    pressure = X[..., 3]

    common = torch.sin(2 * torch.pi * x)
    branch_effect = torch.where(
        process_type == 0,
        0.8 * (temperature - 0.5),
        -0.7 * (pressure - 0.5),
    )
    return (common + branch_effect).unsqueeze(-1)

n_per_branch = 14
x0 = torch.rand(n_per_branch, 1)
x1 = torch.rand(n_per_branch, 1)

X_branch0 = torch.cat([
    x0,
    torch.zeros(n_per_branch, 1),
    torch.rand(n_per_branch, 1),
    torch.rand(n_per_branch, 1),  # この枝では無効
], dim=-1)

X_branch1 = torch.cat([
    x1,
    torch.ones(n_per_branch, 1),
    torch.rand(n_per_branch, 1),  # この枝では無効
    torch.rand(n_per_branch, 1),
], dim=-1)

train_X = torch.cat([X_branch0, X_branch1], dim=0)
train_Y = objective(train_X) + 0.03 * torch.randn(2 * n_per_branch, 1)

print(train_X.shape, train_Y.shape)

## 4. 単一タスク階層型GP

In [ ]:
model = HierarchicalConditionalKernelGP(
    train_X=train_X,
    train_Y=train_Y,
    hierarchical_dependencies=hierarchical_dependencies,
    use_saas_prior=False,
)

print(type(model).__name__)
print('supports_mll:', model.supports_mll)
print('raw_train_X:', model.raw_train_X.shape)
print('raw_train_Y:', model.raw_train_Y.shape)
print('raw_train_Yvar:', model.raw_train_Yvar)

### 学習

階層型GPも robotorchan のexact GP共通インターフェースを使えるため、`make_mll()` から通常どおり学習できます。

In [ ]:
mll = model.make_mll()
fit_gpytorch_mll(mll)
model.eval()

## 5. 枝ごとのposteriorを比較する

方式0では温度を変化させ、圧力は固定します。方式1では逆に圧力を変化させ、温度を固定します。

In [ ]:
grid = torch.linspace(0.0, 1.0, 101)
x_fixed = torch.full_like(grid, 0.35)

X_test0 = torch.stack([
    x_fixed,
    torch.zeros_like(grid),
    grid,
    torch.full_like(grid, 0.5),
], dim=-1)

X_test1 = torch.stack([
    x_fixed,
    torch.ones_like(grid),
    torch.full_like(grid, 0.5),
    grid,
], dim=-1)

with torch.no_grad():
    post0 = model.posterior(X_test0)
    post1 = model.posterior(X_test1)

mean0 = post0.mean.squeeze(-1)
mean1 = post1.mean.squeeze(-1)

plt.figure(figsize=(8, 4))
plt.plot(grid, mean0, label='方式0: 温度を変更')
plt.plot(grid, mean1, label='方式1: 圧力を変更')
plt.xlabel('有効な枝固有変数')
plt.ylabel('posterior mean')
plt.legend()
plt.show()

## 6. 階層型Multi-Task GP

次に、同じ階層探索空間を2つの関連タスクで共有します。

ここでは最後の列をtask featureとして追加します。`hierarchical_dependencies` の列番号は **task featureを除いた後の特徴量** に対する番号なので、先ほどと同じ辞書をそのまま使えます。

In [ ]:
task0 = torch.zeros(train_X.shape[0], 1)
task1 = torch.ones(train_X.shape[0], 1)

train_X_mt = torch.cat([
    torch.cat([train_X, task0], dim=-1),
    torch.cat([train_X, task1], dim=-1),
], dim=0)

train_Y_mt = torch.cat([
    train_Y,
    0.75 * train_Y + 0.20,
], dim=0)

mt_model = HierarchicalConditionalKernelMultiTaskGP(
    train_X=train_X_mt,
    train_Y=train_Y_mt,
    task_feature=4,
    hierarchical_dependencies=hierarchical_dependencies,
    use_saas_prior=False,
)

print('raw_train_X:', mt_model.raw_train_X.shape)
print('raw_train_Y:', mt_model.raw_train_Y.shape)
print('supports_mll:', mt_model.supports_mll)

In [ ]:
mt_mll = mt_model.make_mll()
fit_gpytorch_mll(mt_mll)
mt_model.eval()

## 7. Multi-Task posterior

方式0・温度0.8という同じ条件について、task 0 / task 1 のposteriorを比較します。

In [ ]:
x_grid = torch.linspace(0.0, 1.0, 101)
base = torch.stack([
    x_grid,
    torch.zeros_like(x_grid),
    torch.full_like(x_grid, 0.8),
    torch.full_like(x_grid, 0.5),
], dim=-1)

X_task0 = torch.cat([base, torch.zeros(len(base), 1)], dim=-1)
X_task1 = torch.cat([base, torch.ones(len(base), 1)], dim=-1)

with torch.no_grad():
    mean_task0 = mt_model.posterior(X_task0).mean.squeeze(-1)
    mean_task1 = mt_model.posterior(X_task1).mean.squeeze(-1)

plt.figure(figsize=(8, 4))
plt.plot(x_grid, mean_task0, label='task 0')
plt.plot(x_grid, mean_task1, label='task 1')
plt.xlabel('共通変数 x')
plt.ylabel('posterior mean')
plt.legend()
plt.show()

## 8. 使い分け

### `HierarchicalConditionalKernelGP` が向く場合

- 装置方式によって設定可能なパラメータが変わる
- 材料種によって有効な配合条件が変わる
- アルゴリズム選択によってサブハイパーパラメータが変わる
- 通常の連続・カテゴリ混合だけでは表現しづらい条件付き探索空間

### `HierarchicalConditionalKernelMultiTaskGP` が向く場合

上記に加えて、製品・ライン・装置・測定条件など複数タスク間の相関も同時に利用したい場合に向きます。

### 注意点

`hierarchical_dependencies` は探索空間の構造そのものを表します。単に相関が弱そうな変数を階層化するのではなく、『ある親条件のときだけ子変数に意味がある』という明確な条件付き構造がある場合に使います。

## 9. ベイズ最適化への接続

モデル学習後は通常のBoTorch獲得関数に接続できます。ただし候補生成時にも階層制約を守る必要があります。

たとえば方式0の候補では圧力を無効値に固定し、方式1では温度を固定する、といった候補生成ルールを用意します。探索空間の階層構造と獲得関数最適化の制約は別物なので、両方を整合させることが重要です。